In [ ]:
import os
import pandas as pd
import openai
from langchain.llms import OpenAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from dotenv import load_dotenv

load_dotenv('.env')
OPENAI_API_KEY: str = os.getenv("OPENAI_API_KEY")

: 

In [63]:
job = "BMW"
location = "Mexico"
csv_path = "job_details_{}.csv".format(job)
df_skill = pd.read_csv(csv_path)

In [64]:
df_skill.head()

,state,id,title,description,url,applyMethod,company,location,type,applies,views,expireAt,formattedJobFunctions,jobFunctions,industries,formattedIndustries,formattedExperienceLevel,listedAt,originalListedAt,skills
0,LISTED,3862127651,LAUNCH AND CHANGE COORDINATOR / BATTERY MODULE,"At the BMW Group, everything starts with passi...",https://www.linkedin.com/jobs/view/3862127651/,{'companyApplyUrl': 'https://www.bmwgroup.jobs...,"{'id': 2387, 'name': 'BMW Group', 'universalNa...","San Luis Potosí, Mexico",Full-time,347,2167,1716153576000,"['Manufacturing', 'Production', 'Project Manag...","['MNFC', 'PROD', 'PRJM']","[147, 53, 25]","['Automation Machinery Manufacturing', 'Motor ...",Mid-Senior level,1713357178000,1710969576000,"['Automotive', 'Business English', 'Change Man..."
1,LISTED,3905586613,Coord. Special Sales Development & Management,MAKE LIFE A RIDE.SHARE YOUR PASSION.\nAt the B...,https://www.linkedin.com/jobs/view/3905586613/,"{'companyApplyUrl': '', 'easyApplyUrl': 'https...","{'id': 2387, 'name': 'BMW Group', 'universalNa...","Mexico City, Mexico",Full-time,237,1212,1716586577000,"['Business Development', 'Sales', 'Strategy/Pl...","['BD', 'SALE', 'STRA']",[53],['Motor Vehicle Manufacturing'],Associate,1713994577000,1713994577000,"['Automobiles', 'Communication', 'Automotive',..."
2,LISTED,3907471746,Coord. Product & Price,TODAY'S EXCELLENCE SETS TOMORROW'S STANDARDS.S...,https://www.linkedin.com/jobs/view/3907471746/,"{'companyApplyUrl': '', 'easyApplyUrl': 'https...","{'id': 2387, 'name': 'BMW Group', 'universalNa...","Mexico City, Mexico",Full-time,217,1132,1716732681000,"['Business Development', 'Product Management',...","['BD', 'PRDM', 'SALE']",[53],['Motor Vehicle Manufacturing'],Associate,1714140681000,1714140681000,"['After-Sales', 'Automobiles', 'Automotive', '..."
3,LISTED,3906640920,Sen. Spec. Preemptive & Administrative Collect...,A GOOD INTERNSHIP IS NEVER HANDS OFF.CREATE JO...,https://www.linkedin.com/jobs/view/3906640920/,"{'companyApplyUrl': '', 'easyApplyUrl': 'https...","{'id': 2387, 'name': 'BMW Group', 'universalNa...","Mexico City, Mexico",Full-time,174,1174,1716391906000,"['Administrative', 'Finance', 'Sales']","['ADM', 'FIN', 'SALE']",[53],['Motor Vehicle Manufacturing'],Associate,1713799906000,1713799906000,"['Microsoft Excel', 'Microsoft PowerPoint']"
4,LISTED,3905577019,PROCESS TECHNOLOGY & COMMISSIONING SENIOR SPEC...,"At the BMW Group, everything starts with passi...",https://www.linkedin.com/jobs/view/3905577019/,{'companyApplyUrl': 'https://www.bmwgroup.jobs...,"{'id': 2387, 'name': 'BMW Group', 'universalNa...","San Luis Potosí, Mexico",Full-time,37,439,1716581823000,"['Project Management', 'Production']","['PRJM', 'PROD']",[25],['Manufacturing'],Mid-Senior level,1713989823000,1713989823000,"['5S', 'Business English', 'Continuous Improve..."


In [65]:
job_skills = []
job_softskills = []

llm = OpenAI(model_name="gpt-3.5-turbo-instruct", temperature = 0)
#Add output key to it
template = """ From the given document: {document}
Question: {question}"""
prompt = PromptTemplate(template=template, input_variables=["question", "document"])
llm_chain = LLMChain(prompt=prompt, llm=llm)

for x in df_skill["description"]:
    if len(str(x)) > 3:
        question = "Enumere todas las habilidades mencionadas en el texto. No menciones las habilidades suaves."
        output_skills = llm_chain.run(question = question, document = x)
        job_skills.append(output_skills)
        question = "Enumere todas las habilidades suaves mencionadas en el texto."
        output_softskills = llm_chain.run(question = question, document = x)
        job_softskills.append(output_softskills)    
    else:
        print("description not found")
        job_skills.append("description not found")
        job_softskills.append("description not found")   

In [66]:
df_skill['Skills_extracted'] = job_skills
df_skill['SoftSkills_extracted'] = job_softskills

In [67]:

combine_df_path = "C:/Users/Admin/Documents/VScode/linkedin_scraper/Linkedin_April/{}_completedataset_April_{}.csv".format(job, location)

# Translation

In [68]:
#!pip install googletrans==4.0.0-rc1
#!pip install langdetect

In [69]:
from translate_text import *
import httpcore
setattr(httpcore, 'SyncHTTPTransport', 'AsyncHTTPProxy')

lang = detect_text_lang(df_skill["Skills_extracted"])
df_skill["lang"] = lang
to_span = translate_Eng_to_Span(df_skill["Skills_extracted"])
df_skill["Skills_es"] = to_span
to_span = translate_Eng_to_Span(df_skill["SoftSkills_extracted"])
df_skill["SoftSkills_es"] = to_span

In [70]:
df_skill.head()

,state,id,title,description,url,applyMethod,company,location,type,applies,...,formattedIndustries,formattedExperienceLevel,listedAt,originalListedAt,skills,Skills_extracted,SoftSkills_extracted,lang,Skills_es,SoftSkills_es
0,LISTED,3862127651,LAUNCH AND CHANGE COORDINATOR / BATTERY MODULE,"At the BMW Group, everything starts with passi...",https://www.linkedin.com/jobs/view/3862127651/,{'companyApplyUrl': 'https://www.bmwgroup.jobs...,"{'id': 2387, 'name': 'BMW Group', 'universalNa...","San Luis Potosí, Mexico",Full-time,347,...,"['Automation Machinery Manufacturing', 'Motor ...",Mid-Senior level,1713357178000,1710969576000,"['Automotive', 'Business English', 'Change Man...","\n\n1. Bachelor's degree in Industrial, Mechan...",\n\n1. Pasión\n2. Trabajo en equipo\n3. Lidera...,en,"1. Licenciatura en ingeniería industrial, mecá...",\n\n1. Pasión\n2. Trabajo en equipo\n3. Lidera...
1,LISTED,3905586613,Coord. Special Sales Development & Management,MAKE LIFE A RIDE.SHARE YOUR PASSION.\nAt the B...,https://www.linkedin.com/jobs/view/3905586613/,"{'companyApplyUrl': '', 'easyApplyUrl': 'https...","{'id': 2387, 'name': 'BMW Group', 'universalNa...","Mexico City, Mexico",Full-time,237,...,['Motor Vehicle Manufacturing'],Associate,1713994577000,1713994577000,"['Automobiles', 'Communication', 'Automotive',...",\n1. Conocimiento y pasión por la industria au...,\n1. Pasión\n2. Trabajo en equipo\n3. Motivaci...,es,\n1. Conocimiento y pasión por la industria au...,\n1. Pasión\n2. Trabajo en equipo\n3. Motivaci...
2,LISTED,3907471746,Coord. Product & Price,TODAY'S EXCELLENCE SETS TOMORROW'S STANDARDS.S...,https://www.linkedin.com/jobs/view/3907471746/,"{'companyApplyUrl': '', 'easyApplyUrl': 'https...","{'id': 2387, 'name': 'BMW Group', 'universalNa...","Mexico City, Mexico",Full-time,217,...,['Motor Vehicle Manufacturing'],Associate,1714140681000,1714140681000,"['After-Sales', 'Automobiles', 'Automotive', '...",\n\n1. Product planning\n2. Launch management\...,\n\n1. Pasión\n2. Dedicación\n3. Creatividad\n...,en,1. Planificación de productos\n2. Gestión de l...,\n\n1. Pasión\n2. Dedicación\n3. Creatividad\n...
3,LISTED,3906640920,Sen. Spec. Preemptive & Administrative Collect...,A GOOD INTERNSHIP IS NEVER HANDS OFF.CREATE JO...,https://www.linkedin.com/jobs/view/3906640920/,"{'companyApplyUrl': '', 'easyApplyUrl': 'https...","{'id': 2387, 'name': 'BMW Group', 'universalNa...","Mexico City, Mexico",Full-time,174,...,['Motor Vehicle Manufacturing'],Associate,1713799906000,1713799906000,"['Microsoft Excel', 'Microsoft PowerPoint']",\n1. Conocimiento en finanzas\n2. Experiencia ...,\n1. Trabajo en equipo\n2. Colaboración intern...,es,\n1. Conocimiento en finanzas\n2. Experiencia ...,\n1. Trabajo en equipo\n2. Colaboración intern...
4,LISTED,3905577019,PROCESS TECHNOLOGY & COMMISSIONING SENIOR SPEC...,"At the BMW Group, everything starts with passi...",https://www.linkedin.com/jobs/view/3905577019/,{'companyApplyUrl': 'https://www.bmwgroup.jobs...,"{'id': 2387, 'name': 'BMW Group', 'universalNa...","San Luis Potosí, Mexico",Full-time,37,...,['Manufacturing'],Mid-Senior level,1713989823000,1713989823000,"['5S', 'Business English', 'Continuous Improve...",\n\n1. Engineering experience\n2. Commissionin...,\n\n1. Pasión\n2. Trabajo en equipo\n3. Colabo...,en,1. Experiencia de ingeniería\n2. Coss puros de...,\n\n1. Pasión\n2. Trabajo en equipo\n3. Colabo...


In [71]:
df_skill.to_csv(combine_df_path)